# 🏥 Facial Skin Disease Classification: Deep Transfer Learning & Ensemble Benchmark
### A Multi-Architecture Study: EfficientNetV2 · ResNet-50 · DenseNet-121 · MobileNetV3 ·

In [ ]:
import subprocess, sys

def pip_install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

pip_install('keras-tuner')
pip_install('scikit-learn')
pip_install('seaborn')
pip_install('imbalanced-learn')

print('✅ Dependencies installed.')

In [ ]:
# ─── Standard Library ────────────────────────────────────────────────────────
import os, gc, json, warnings, random, math, shutil
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

# ─── Numerical & Visualisation ───────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ─── Sklearn ─────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    cohen_kappa_score, matthews_corrcoef, f1_score
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

# ─── TensorFlow / Keras ──────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.applications import (
    EfficientNetV2B0,
    ResNet50,
    DenseNet121,
    MobileNetV3Large
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau,
    ModelCheckpoint, TensorBoard, CSVLogger
)
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts
from tensorflow.keras.regularizers import l2

# ─── Seed Everything ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'✅ TensorFlow {tf.__version__}')
print(f'✅ GPUs available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ─── Dataset Path ────────────────────────────────────────────────────────────
# Kaggle: update this path to match your dataset location
DATA_DIR = Path('')   # ← change if your dataset folder name differs

# ─── Training Hypers ─────────────────────────────────────────────────────────
IMG_SIZE             = 224
BATCH_SIZE           = 32
NUM_CLASSES          = 6
EPOCHS_PHASE1        = 15
EPOCHS_PHASE2        = 25
EPOCHS_PHASE3        = 30
BASE_LR              = 1e-3
FINETUNE_LR          = 1e-4
FULL_FT_LR           = 2e-5
DROPOUT_RATE         = 0.4
L2_REG               = 1e-4
MAX_IMAGES_PER_CLASS = 1500
TARGET_SIZE_BALANCE  = 1200
SAVE_DIR             = Path('/kaggle/working/saved_models')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR             = Path('/kaggle/working/plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Configuration set.')

In [ ]:
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

def discover_dataset(data_dir: Path):
    """Walk the directory tree, collect (path, label) pairs, report stats."""
    assert data_dir.exists(), f'❌ DATA_DIR not found: {data_dir}'
    classes = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
    assert len(classes) > 0, '❌ No sub-folders found in DATA_DIR.'

    records, bad_files = [], []
    for label_idx, cls in enumerate(classes):
        cls_dir = data_dir / cls
        for f in cls_dir.rglob('*'):
            if f.suffix.lower() in VALID_EXTS:
                records.append({'path': str(f), 'label': label_idx, 'class': cls})
            elif f.is_file():
                bad_files.append(str(f))

    df = pd.DataFrame(records)
    print(f'\n📊 Dataset Summary')
    print(f'  Total valid images : {len(df)}')
    print(f'  Non-image files    : {len(bad_files)}')
    print(f'  Classes ({len(classes)})        : {classes}\n')
    print(df.groupby('class')['path'].count().rename('count').to_string())
    return df, classes

df_full, CLASS_NAMES = discover_dataset(DATA_DIR)
print(f'\n✅ {len(CLASS_NAMES)} classes discovered.')

In [ ]:
from PIL import Image as PILImage

def is_valid_image(path: str, min_size: int = 32) -> bool:
    """Fully decode image with PIL to catch any corrupt or unreadable files."""
    try:
        with PILImage.open(path) as img:
            img.load()          # forces full decode — not just header check
            w, h = img.size
            if w < min_size or h < min_size:
                return False
            img.convert('RGB') # ensures it is RGB-convertible
        return True
    except Exception:
        return False

print('🔍 Scanning for corrupt / invalid images (may take ~2-3 min)...')
valid_mask = df_full['path'].apply(is_valid_image)
n_bad      = (~valid_mask).sum()
df_clean   = df_full[valid_mask].reset_index(drop=True)

print(f'  Removed corrupt/invalid images : {n_bad}')
print(f'  Clean dataset size             : {len(df_clean)}')
print('✅ Cleaning done.')

In [ ]:
# ─── Prompt-ID grouped split before balancing (prevents data leakage) ────────
# First 5 synthetic classes: the PROMPT ID is the splitting unit, but prompt
# IDs are only unique WITHIN a class (e.g. prompt "065" exists independently
# in Acne, Vitiligo, Normal Skin, etc. — confirmed from real data). So the
# grouping key must be class + prompt_id together. All 5 variants of a given
# (class, prompt_id) pair are assigned to exactly ONE partition (70% of prompt
# groups -> Train, 15% -> Validation, 15% -> Test), so no prompt's variants
# can cross the train/val/test boundary.
# Skin Cancer: the BCC / SCC / MEL subtypes do not follow the prompt/variant
# structure, so they continue to be split independently at the IMAGE level
# (70/15/15) and are recombined under the single existing "Skin Cancer" class
# label. This happens after corruption filtering (df_clean) and before the
# existing train-only balancing (Section 7).

import re

_PROMPT_RE = re.compile(r'_p_(\d+)_v_(\d+)$', re.IGNORECASE)
_CANCER_RE = re.compile(r'_(BCC|SCC|MEL)_(\d+)$', re.IGNORECASE)

def _parse_stem(path_str: str):
    """Classify a filename as a synthetic prompt/variant image or a
    Skin-Cancer BCC/SCC/MEL image, ignoring the extension."""
    stem = Path(path_str).stem
    m = _PROMPT_RE.search(stem)
    if m:
        return 'synthetic', m.group(1), int(m.group(2))
    m = _CANCER_RE.search(stem)
    if m:
        return 'cancer', m.group(1).upper(), int(m.group(2))
    return 'unknown', None, None

_kinds, _keys, _variants = [], [], []
for p in df_clean['path']:
    kind, key, var = _parse_stem(p)
    _kinds.append(kind); _keys.append(key); _variants.append(var)

df_clean = df_clean.copy()
df_clean['file_kind'] = _kinds                                          # 'synthetic' | 'cancer' | 'unknown'
df_clean['prompt_id'] = [k if kd == 'synthetic' else None for kd, k in zip(_kinds, _keys)]
df_clean['variant']   = [v if kd == 'synthetic' else None for kd, v in zip(_kinds, _variants)]
df_clean['subtype']   = [k if kd == 'cancer'    else None for kd, k in zip(_kinds, _keys)]

# group_key = class + prompt_id — REQUIRED because prompt IDs repeat across
# classes (e.g. "065" independently exists in Acne, Vitiligo, Normal Skin...).
# Without the class prefix, unrelated images from 5 different classes get
# merged into one fake 25-image "group".
df_clean['group_key'] = [
    f'{cls}_{pid}' if kd == 'synthetic' else None
    for kd, cls, pid in zip(_kinds, df_clean['class'], df_clean['prompt_id'])
]

n_unknown = (df_clean['file_kind'] == 'unknown').sum()
if n_unknown:
    print(f'⚠️  {n_unknown} image(s) matched neither the prompt/variant nor BCC/SCC/MEL '
          f'filename pattern and will be EXCLUDED from the split:')
    print(df_clean.loc[df_clean['file_kind'] == 'unknown', 'path'].head(15).to_string(index=False))

# ── First 5 synthetic classes: PROMPT-GROUPED 70/15/15 split ─────────────────
df_synth = df_clean[df_clean['file_kind'] == 'synthetic']

# Completeness = exactly 5 DISTINCT variant values for this (class, prompt_id)
# pair — not literally [1,2,3,4,5], since numbering conventions can vary.
complete_group_keys, incomplete_report = [], []
for gk in sorted(df_synth['group_key'].unique()):
    grp = df_synth[df_synth['group_key'] == gk]
    variants = sorted(int(v) for v in grp['variant'])
    if len(grp) == 5 and len(set(variants)) == 5:
        complete_group_keys.append(gk)
    else:
        n_dupe = len(grp) - len(set(variants))
        incomplete_report.append({
            'group_key': gk, 'n_images': len(grp),
            'variants_present': variants, 'duplicates': n_dupe
        })

complete_group_keys = sorted(complete_group_keys)   # deterministic order before shuffling

print(f'\n📋 Synthetic (first 5 classes) prompt-group validation:')
print(f'  Complete groups (5/5)     : {len(complete_group_keys)}')
print(f'  Incomplete/duplicate      : {len(incomplete_report)}  (excluded from split)')
if incomplete_report:
    print('  Excluded groups (showing up to 15):')
    for r in incomplete_report[:15]:
        print(f"    {r['group_key']}: {r['n_images']} images, "
              f"variants={r['variants_present']}, duplicates={r['duplicates']}")

assert len(complete_group_keys) > 0, (
    '❌ No complete prompt groups found — check the "variants_present" values '
    'printed above against what a genuinely complete group should look like.'
)

# Prompt GROUPS (not images) are split 70% / 15% / 15%, deterministically.
# All 5 variants of a given prompt travel together into exactly one partition.
train_groups, temp_groups = train_test_split(
    complete_group_keys, test_size=0.30, random_state=SEED, shuffle=True)
val_groups, test_groups = train_test_split(
    temp_groups, test_size=0.50, random_state=SEED, shuffle=True)

train_groups_set, val_groups_set, test_groups_set = set(train_groups), set(val_groups), set(test_groups)

synth_train_idx = df_synth.index[df_synth['group_key'].isin(train_groups_set)].tolist()
synth_val_idx   = df_synth.index[df_synth['group_key'].isin(val_groups_set)].tolist()
synth_test_idx  = df_synth.index[df_synth['group_key'].isin(test_groups_set)].tolist()

# ── Skin Cancer: BCC / SCC / MEL split independently 70/15/15 (image level) ──
df_cancer = df_clean[df_clean['file_kind'] == 'cancer']
cancer_train_idx, cancer_val_idx, cancer_test_idx = [], [], []
cancer_report = {}

for subtype in sorted(df_cancer['subtype'].unique()):
    sub     = df_cancer[df_cancer['subtype'] == subtype]
    idx_all = sub.index.to_numpy()
    if len(idx_all) < 3:
        # Too few images to meaningfully split — keep them all in train.
        cancer_train_idx.extend(idx_all)
        cancer_report[subtype] = (len(idx_all), 0, 0)
        continue
    idx_train, idx_temp = train_test_split(idx_all,  test_size=0.30, random_state=SEED)
    idx_val,   idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED)
    cancer_train_idx.extend(idx_train)
    cancer_val_idx.extend(idx_val)
    cancer_test_idx.extend(idx_test)
    cancer_report[subtype] = (len(idx_train), len(idx_val), len(idx_test))

# ── Combine synthetic + cancer index sets into the final path/label arrays ───
train_idx = list(synth_train_idx) + list(cancer_train_idx)
val_idx   = list(synth_val_idx)   + list(cancer_val_idx)
test_idx  = list(synth_test_idx)  + list(cancer_test_idx)

X_train_raw = df_clean.loc[train_idx, 'path'].values
y_train_raw = df_clean.loc[train_idx, 'label'].values
X_val       = df_clean.loc[val_idx,   'path'].values
y_val       = df_clean.loc[val_idx,   'label'].values
X_test      = df_clean.loc[test_idx,  'path'].values
y_test      = df_clean.loc[test_idx,  'label'].values

# ── Leakage validation #1: no image PATH in more than one partition ──────────
set_train, set_val, set_test = set(X_train_raw), set(X_val), set(X_test)
assert not (set_train & set_val),  '❌ Leakage detected: train ∩ val is non-empty!'
assert not (set_train & set_test), '❌ Leakage detected: train ∩ test is non-empty!'
assert not (set_val & set_test),   '❌ Leakage detected: val ∩ test is non-empty!'

# ── Leakage validation #2: no synthetic PROMPT_ID in more than one partition ─
assert not (train_groups_set & val_groups_set),  '❌ Leakage detected: a prompt_id is in both train and val!'
assert not (train_groups_set & test_groups_set), '❌ Leakage detected: a prompt_id is in both train and test!'
assert not (val_groups_set & test_groups_set),   '❌ Leakage detected: a prompt_id is in both val and test!'

# Sanity check: every complete group landed ENTIRELY in exactly one partition
# (all 5 of its images), never split across partitions.
group_alloc_ok = True
for gk in complete_group_keys:
    grp = df_synth[df_synth['group_key'] == gk]
    n_tr = grp.index.isin(train_idx).sum()
    n_va = grp.index.isin(val_idx).sum()
    n_te = grp.index.isin(test_idx).sum()
    buckets_used = sum(n > 0 for n in (n_tr, n_va, n_te))
    total = n_tr + n_va + n_te
    if buckets_used != 1 or total != 5:
        group_alloc_ok = False

# ── Skin Cancer per-subtype allocation validation ─────────────────────────────
cancer_alloc_ok = True
for subtype, (ntr, nva, nte) in cancer_report.items():
    n_total = len(df_cancer[df_cancer['subtype'] == subtype])
    if n_total >= 3 and (ntr + nva + nte) != n_total:
        cancer_alloc_ok = False

print('\n' + '=' * 60)
print('PROMPT-ID GROUPED DATA SPLIT')
print('=' * 60)

n_groups = len(complete_group_keys)
print(f'\nFirst 5 synthetic classes:')
print(f'  Complete prompt groups : {n_groups}')
print(f'  Train groups : {len(train_groups)}  ({len(train_groups)/n_groups:.1%})  -> {len(synth_train_idx)} images')
print(f'  Val groups   : {len(val_groups)}  ({len(val_groups)/n_groups:.1%})  -> {len(synth_val_idx)} images')
print(f'  Test groups  : {len(test_groups)}  ({len(test_groups)/n_groups:.1%})  -> {len(synth_test_idx)} images')

print(f'\nSkin Cancer (image-level 70/15/15):')
for subtype in ['BCC', 'SCC', 'MEL']:
    if subtype in cancer_report:
        ntr, nva, nte = cancer_report[subtype]
        print(f'  {subtype}  -> Train: {ntr} | Val: {nva} | Test: {nte}')

print(f'\nOverall:')
print(f'  Train: {len(X_train_raw)}')
print(f'  Validation: {len(X_val)}')
print(f'  Test: {len(X_test)}')

print(f'\nLeakage check (image paths):')
print(f'  Train ∩ Validation: {len(set_train & set_val)}')
print(f'  Train ∩ Test: {len(set_train & set_test)}')
print(f'  Validation ∩ Test: {len(set_val & set_test)}')

print(f'\nLeakage check (prompt_id groups):')
print(f'  Train ∩ Validation: {len(train_groups_set & val_groups_set)}')
print(f'  Train ∩ Test: {len(train_groups_set & test_groups_set)}')
print(f'  Validation ∩ Test: {len(val_groups_set & test_groups_set)}')

print(f'\nGroup allocation check (every prompt entirely in ONE partition):')
print(f'  {"PASS" if (group_alloc_ok and cancer_alloc_ok) else "FAIL"}')
assert group_alloc_ok,  '❌ One or more prompt groups were split across partitions!'
assert cancer_alloc_ok, '❌ One or more Skin Cancer subtypes did not fully allocate!'

print('\n✅ Split complete — no path and no prompt_id crosses a partition boundary.')

In [ ]:
def balance_dataset(df: pd.DataFrame, target: int, max_cap: int, seed: int = 42):
    parts = []
    for cls_name, grp in df.groupby('class'):
        n = len(grp)
        if n < target:
            extra = grp.sample(target - n, replace=True, random_state=seed)
            parts.append(pd.concat([grp, extra]))
        elif n > max_cap:
            parts.append(grp.sample(max_cap, random_state=seed))
        else:
            parts.append(grp)
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

# ─── Balance ONLY the training fold — val/test stay at real-world distribution ──
idx_to_class = {i: c for i, c in enumerate(CLASS_NAMES)}
train_df_raw = pd.DataFrame({
    'path' : X_train_raw,
    'label': y_train_raw,
    'class': [idx_to_class[l] for l in y_train_raw]
})

# Target = mean class count within the TRAIN split itself (computed post-split,
# so it reflects only the 70% partition, never touching val/test images).
train_counts = train_df_raw['class'].value_counts()
train_target = int(round(train_counts.mean()))

df_balanced = balance_dataset(train_df_raw, target=train_target, max_cap=train_target)

print('\n📊 Balanced class distribution (TRAIN split only):')
print(df_balanced.groupby('class')['path'].count().rename('count').to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train_df_raw.groupby('class')['path'].count().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='k')
axes[0].set_title('Train Before Balancing', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Image Count'); axes[0].tick_params(axis='x', rotation=45)

df_balanced.groupby('class')['path'].count().plot(
    kind='bar', ax=axes[1], color='seagreen', edgecolor='k')
axes[1].set_title('Train After Balancing', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Image Count'); axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Balancing done (train only — val/test never resampled).')

X_train = df_balanced['path'].values
y_train = df_balanced['label'].values


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# ─── Augmentation layer (mild — preserves medical texture) ───────────────────
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.08),
    layers.RandomContrast(0.10),
    layers.RandomBrightness(0.08),
    layers.RandomTranslation(0.05, 0.05),
], name='augmentation')

# ─── PIL-based decoder: handles JPEG/PNG/BMP/TIFF/WebP/any misnamed file ─────
from PIL import Image as PILImage
import io

def _pil_load(path_bytes, raw_bytes):
    """Load any image format with PIL and return a float32 RGB array in [0, 255].
    Raises a clear error naming the offending file path if the image is
    corrupted/unreadable — no silent black-image fallback."""
    try:
        with PILImage.open(io.BytesIO(raw_bytes)) as img:
            img = img.convert('RGB')
            img = img.resize((IMG_SIZE, IMG_SIZE), PILImage.BILINEAR)
            # Raw [0, 255] pixels are returned here — each backbone needs a
            # different input range (Caffe-style BGR mean-subtract for
            # ResNet50, torch-style mean/std for DenseNet121, internal
            # rescale for EfficientNetV2B0 / MobileNetV3Large). Per-backbone
            # normalisation is applied inside the model itself (see the
            # preprocessing layers defined earlier), so it's baked into the
            # saved .keras file and can never be forgotten at inference time.
            return np.array(img, dtype=np.float32)
    except Exception as e:
        file_path = path_bytes.decode('utf-8') if isinstance(path_bytes, bytes) else str(path_bytes)
        raise RuntimeError(
            f"\u274c Corrupted or unreadable image encountered during preprocessing: '{file_path}' ({e})"
        ) from e

def preprocess_image(path, label, augment=False):
    # Read raw bytes from disk
    raw = tf.io.read_file(path)

    # tf.py_function bridges PIL into the TF graph — handles ALL formats including TIFF
    image = tf.py_function(
        func=lambda p, r: _pil_load(p.numpy(), r.numpy()),
        inp=[path, raw],
        Tout=tf.float32
    )
    # Explicitly set shape so the TF graph knows dimensions downstream
    image.set_shape([IMG_SIZE, IMG_SIZE, 3])

    if augment:
        image = data_augmentation(image, training=True)
    # Range is [0, 255] here on purpose — per-backbone normalisation happens
    # inside the model itself (see Cell 10 / PREPROCESS_LAYERS).
    image = tf.clip_by_value(image, 0.0, 255.0)
    label = tf.cast(label, tf.int32)
    return image, label

def make_dataset(paths, labels, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(paths, dtype=tf.string),
         tf.constant(labels, dtype=tf.int32))
    )
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    # num_parallel_calls=4 (not AUTOTUNE) — tf.py_function needs fixed thread count
    ds = ds.map(lambda p, l: preprocess_image(p, l, augment), num_parallel_calls=4)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(X_train, y_train, augment=True,  shuffle=True)
val_ds   = make_dataset(X_val,   y_val,   augment=False, shuffle=False)
test_ds  = make_dataset(X_test,  y_test,  augment=False, shuffle=False)

print('✅ tf.data pipelines created.')

# ─── Sanity visual ────────────────────────────────────────────────────────────
sample_images, sample_labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    img = sample_images[i].numpy()
    img = np.clip(img / 255.0, 0.0, 1.0)   # display-only rescale — pipeline itself stays [0,255]
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[sample_labels[i].numpy()], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Augmented Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'sample_images.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Loss Function Formulation: Class-Weighted Multi-Class Focal Loss
Standard Cross-Entropy loss suffers in the presence of extreme class imbalance and ambiguous lesion boundaries because easy negative examples dominate the accumulated gradient.

### Multi-Class Focal Loss
We utilize Focal Loss (Lin et al., *RetinaNet*), defined as:
$$\mathcal{L}_{\text{focal}}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

* **Focusing Parameter ($\gamma = 2.0$):** Dynamically scales down the gradient contribution of well-classified easy samples, concentrating model capacity on hard, boundary-case lesions.
* **Alpha Balancing ($\alpha = 0.25$):** Scales minority-class penalization.
* Combined with compute-derived inverse class weights for balanced optimization.

In [ ]:
# ─── Class weights ────────────────────────────────────────────────────────────
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train
)
CLASS_WEIGHT_DICT = {i: float(w) for i, w in enumerate(class_weights_array)}
print('Class weights:', {CLASS_NAMES[k]: round(v, 3) for k, v in CLASS_WEIGHT_DICT.items()})

# ─── Focal Loss ───────────────────────────────────────────────────────────────
class FocalLoss(keras.losses.Loss):
    """
    Multi-class Focal Loss.
    Reference: Lin et al. 2017 (https://arxiv.org/abs/1708.02002)
    """
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true    = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred    = tf.cast(y_pred, tf.float32)
        # The model's final layer already applies softmax, so y_pred arrives
        # as probabilities — clip for numerical stability rather than
        # re-applying softmax, which would distort the gradient.
        y_pred    = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_true_oh = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1], dtype=tf.float32)
        pt        = tf.reduce_sum(y_true_oh * y_pred, axis=-1)
        ce        = -tf.math.log(pt)
        focal     = self.alpha * tf.pow(1.0 - pt, self.gamma) * ce
        return tf.reduce_mean(focal)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma': self.gamma, 'alpha': self.alpha})
        return cfg

print('✅ FocalLoss defined.')


## 10. Architectural Head Construction & Normalization Invariance
To ensure fair and rigorous benchmarking across all five architectures, we employ a unified, standardized classification head:

$$\text{Backbone Features} \longrightarrow \text{GlobalAveragePooling2D} \longrightarrow \text{BatchNormalization} \longrightarrow \text{Dropout}(0.4) \longrightarrow \text{Dense}(128, \text{Mish}) \longrightarrow \text{Dropout}(0.3) \longrightarrow \text{Dense}(K, \text{Softmax})$$

### Normalization Invariance Design
Different pretrained ImageNet backbones require different input scalings:
* **EfficientNetV2 & MobileNetV3:** Built-in normalization layers inside Keras (`include_preprocessing=True`).
* **ResNet-50 & DenseNet-121:** Require Caffe-style mean subtraction and zero-centering via `preprocess_input`.

We encapsulate architecture-specific preprocessing inside custom Keras layers (`ResNet50Preprocess`, `DenseNet121Preprocess`). Thus, the input pipeline supplies uniform raw $[0, 255]$ tensors, ensuring full modularity and zero preprocessing mismatch.

In [ ]:
# ─── Backbone-specific preprocessing layers ──────────────────────────────────
# The pipeline (Cell 8) now hands every backbone the SAME raw [0, 255] float32
# image. Each backbone still needs its own ImageNet-pretraining-matched
# normalisation:
#  - MobileNetV3Large / EfficientNetV2B0: built with include_preprocessing=True,
#    so Keras already bakes the correct rescaling into the model itself — fed
#    raw [0, 255] directly, no extra layer needed.
#  - ResNet50 / DenseNet121: have no such built-in option, so they get an
#    explicit keras.applications.<name>.preprocess_input layer right after
#    the Input layer.
class ResNet50Preprocess(layers.Layer):
    """Caffe-style ResNet50 preprocessing. Expects raw [0, 255] RGB input."""
    def call(self, x):
        return keras.applications.resnet50.preprocess_input(x)


class DenseNet121Preprocess(layers.Layer):
    """Torch-style DenseNet121 preprocessing. Expects raw [0, 255] RGB input."""
    def call(self, x):
        return keras.applications.densenet.preprocess_input(x)


# Maps each backbone name to the preprocessing layer it needs.
# `None` == no extra layer (the backbone's `include_preprocessing=True` already handles it).
PREPROCESS_LAYERS = {
    'ResNet50':          ResNet50Preprocess,
    'DenseNet121':       DenseNet121Preprocess,
    'MobileNetV3Large':  None,
    'EfficientNetV2B0':  None,
}

# Registry every custom layer that gets baked into a saved .keras model.
# ALWAYS pass this (merged with {'FocalLoss': FocalLoss}) to keras.models.load_model,
# for every model — otherwise loading a ResNet50/DenseNet121 checkpoint raises
# "Unknown layer" and silently fails.
CUSTOM_OBJECTS = {
    'ResNet50Preprocess':    ResNet50Preprocess,
    'DenseNet121Preprocess': DenseNet121Preprocess,
}


def get_preprocess_layer(model_name: str):
    """Return a fresh preprocessing layer instance for `model_name`, or None."""
    layer_cls = PREPROCESS_LAYERS.get(model_name)
    return layer_cls(name='backbone_preprocess') if layer_cls is not None else None


def build_classification_head(base_model, num_classes, dropout_rate, l2_reg, name='model',
                               preprocess_layer=None):
    """
    Attach classification head: GAP → BN → Dense(512) → Dropout → Dense(256) → Softmax

    `preprocess_layer`, if given, is applied to the raw [0, 255] input BEFORE it
    reaches `base_model` — pass the result of `get_preprocess_layer(model_name)`.
    """
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_image')
    x = inputs
    if preprocess_layer is not None:
        x = preprocess_layer(x)

    x = base_model(x)

    if len(x.shape) == 4:
        x = layers.GlobalAveragePooling2D(name='gap')(x)
    elif len(x.shape) == 3:
        x = layers.Lambda(lambda t: t[:, 0, :], name='cls_token')(x)

    x = layers.BatchNormalization(name='bn_head')(x)
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=l2(l2_reg), name='dense_512')(x)
    x = layers.BatchNormalization(name='bn_512')(x)
    x = layers.Dropout(dropout_rate, name='drop_512')(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=l2(l2_reg), name='dense_256')(x)
    x = layers.BatchNormalization(name='bn_256')(x)
    x = layers.Dropout(dropout_rate * 0.5, name='drop_256')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    return Model(inputs, outputs, name=name)


def unfreeze_layers(base_model, n_layers: int):
    base_model.trainable = True
    for layer in base_model.layers[:-n_layers]:
        layer.trainable = False
    n_trainable = sum(1 for l in base_model.layers if l.trainable)
    print(f'  → {n_trainable}/{len(base_model.layers)} backbone layers trainable')


def freeze_all(base_model):
    base_model.trainable = False
    print('  → Backbone fully frozen')


print('✅ Model builder utilities ready.')


## 11. Dynamic Training Callbacks & Adaptive Learning Rate Schedules
To stabilize training dynamics and prevent catastrophic overfitting, each model is guided by three complementary callbacks:
1. **`ModelCheckpoint`:** Automatically persists the optimal weights whenever validation accuracy reaches a new peak.
2. **`EarlyStopping`:** Halts training if validation accuracy plateaus for 8 consecutive epochs, instantly restoring best model weights.
3. **`ReduceLROnPlateau`:** Dynamically scales the learning rate by $0.4\times$ when validation loss stalls for 4 epochs, enabling precise parameter convergence.

In [ ]:
def make_callbacks(model_name: str, monitor: str = 'val_accuracy', patience_es: int = 8):
    ckpt_path = str(SAVE_DIR / f'{model_name}_best.keras')
    callbacks = [
        ModelCheckpoint(
            filepath=ckpt_path,
            monitor=monitor, save_best_only=True,
            verbose=1, mode='max'
        ),
        EarlyStopping(
            monitor=monitor, patience=patience_es,
            restore_best_weights=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.4,
            patience=4, min_lr=1e-7, verbose=1
        ),
        CSVLogger(str(SAVE_DIR / f'{model_name}_training_log.csv'))
    ]
    return callbacks, ckpt_path


def cosine_lr_schedule(base_lr, epochs, warmup=3):
    def schedule(epoch):
        if epoch < warmup:
            return float(base_lr * (epoch + 1) / warmup)
        progress = (epoch - warmup) / max(1, epochs - warmup)
        return float(base_lr * 0.5 * (1 + math.cos(math.pi * progress)))
    return keras.callbacks.LearningRateScheduler(schedule, verbose=0)


print('✅ Callbacks ready.')

## 12. Clinical Diagnostic Evaluation Suite & Visualization Engine
Standard classification accuracy is insufficient for clinical dermatological applications. Our evaluation suite computes a comprehensive spectrum of diagnostic metrics on the held-out test set:
* **Multi-Class Accuracy & Balanced Accuracy:** Overall vs class-averaged accuracy.
* **Macro F1 & Weighted F1:** Unweighted harmonic mean (highlighting minority class performance) vs sample-weighted score.
* **Top-2 Categorical Accuracy:** Measures if the true diagnosis is within the differential top-2 candidate list.
* **One-vs-Rest AUC-ROC:** Multi-class discriminative confidence across varying decision thresholds.
* **Cohen's Kappa ($\kappa$) & Matthews Correlation Coefficient (MCC):** Chance-adjusted inter-rater reliability metrics.
* **Visualizations:** Normalized Confusion Matrices and Phase-wise Learning Curves.

In [ ]:
def evaluate_model(model, test_ds, class_names, model_name):
    """Compute comprehensive evaluation metrics."""
    y_true, y_pred_prob = [], []
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred_prob.extend(preds)

    y_true      = np.array(y_true)
    y_pred_prob = np.array(y_pred_prob)
    y_pred      = np.argmax(y_pred_prob, axis=1)

    acc    = (y_pred == y_true).mean()
    f1_mac = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    f1_wt  = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    kappa  = cohen_kappa_score(y_true, y_pred)
    mcc    = matthews_corrcoef(y_true, y_pred)
    report = classification_report(y_true, y_pred,
                                    target_names=class_names,
                                    zero_division=0, output_dict=True)

    y_bin = label_binarize(y_true, classes=np.arange(len(class_names)))
    try:
        auc_roc = roc_auc_score(y_bin, y_pred_prob, average='macro', multi_class='ovr')
    except Exception:
        auc_roc = float('nan')

    return {
        'model'      : model_name,
        'accuracy'   : round(float(acc), 4),
        'f1_macro'   : round(float(f1_mac), 4),
        'f1_weighted': round(float(f1_wt), 4),
        'kappa'      : round(float(kappa), 4),
        'mcc'        : round(float(mcc), 4),
        'auc_roc'    : round(float(auc_roc), 4),
        'report'     : report,
        'y_true'     : y_true,
        'y_pred'     : y_pred,
        'y_pred_prob': y_pred_prob,
    }


def plot_confusion_matrix(y_true, y_pred, class_names, model_name, save_dir):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title(f'{model_name} — Confusion Matrix (Counts)', fontweight='bold')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
                xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title(f'{model_name} — Confusion Matrix (Normalised)', fontweight='bold')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

    plt.tight_layout()
    plt.savefig(save_dir / f'{model_name}_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_roc_curves(y_true, y_pred_prob, class_names, model_name, save_dir):
    n_classes = len(class_names)
    y_bin     = label_binarize(y_true, classes=np.arange(n_classes))

    fig, ax = plt.subplots(figsize=(10, 7))
    colors  = plt.cm.tab10(np.linspace(0, 1, n_classes))
    for i, (cls, col) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_prob[:, i])
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=col, lw=2, label=f'{cls} (AUC={roc_auc_val:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{model_name} — ROC Curves (One-vs-Rest)', fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / f'{model_name}_roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_training_history(history_dict, model_name, save_dir):
    merged = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}
    for h in history_dict:
        for k in merged:
            if k in h:
                merged[k].extend(h[k])

    epochs = range(1, len(merged['accuracy']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, merged['accuracy'],     label='Train Acc', lw=2)
    axes[0].plot(epochs, merged['val_accuracy'], label='Val Acc',   lw=2, linestyle='--')
    axes[0].set_title(f'{model_name} — Accuracy', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, merged['loss'],     label='Train Loss', lw=2)
    axes[1].plot(epochs, merged['val_loss'], label='Val Loss',   lw=2, linestyle='--')
    axes[1].set_title(f'{model_name} — Loss', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.suptitle(f'{model_name} Training Curves', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_dir / f'{model_name}_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_per_class_metrics(report_dict, class_names, model_name, save_dir):
    metrics_list = ['precision', 'recall', 'f1-score']
    data = {m: [report_dict[c][m] for c in class_names] for m in metrics_list}
    df_m = pd.DataFrame(data, index=class_names)

    ax = df_m.plot(kind='bar', figsize=(12, 5), width=0.7, edgecolor='k',
                   color=['#4C72B0', '#55A868', '#C44E52'])
    ax.set_title(f'{model_name} — Per-Class Metrics', fontsize=13, fontweight='bold')
    ax.set_ylabel('Score'); ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(loc='lower right')
    ax.grid(axis='y', alpha=0.3)
    for bar in ax.patches:
        ax.annotate(f'{bar.get_height():.2f}',
                    (bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01),
                    ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(save_dir / f'{model_name}_per_class_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()


print('✅ Evaluation utilities ready.')

## 13. Progressive 3-Phase Transfer Learning Strategy
Training deep backbones on specialized medical imagery requires careful gradient control to avoid destroying pretrained ImageNet weights (*catastrophic forgetting*).

We employ a structured **3-Phase Progressive Unfreezing** protocol:

```
┌────────────────────────────────────────────────────────────────────────┐
│  Phase 1: Feature Extraction / Head Warmup                             │
│  - Backbone: 100% FROZEN                                               │
│  - Classification Head: TRAINABLE (LR = 1e-3)                          │
│  - Goal: Warm up classification head to prevent destructive gradients  │
└──────────────────────────────────┬─────────────────────────────────────┘
                                   ▼
┌────────────────────────────────────────────────────────────────────────┐
│  Phase 2: Partial Unfreezing (Top Blocks)                              │
│  - Backbone: Top N layers UNFROZEN                                     │
│  - Learning Rate: Scaled down (LR = 1e-4)                              │
│  - Goal: Adapt high-level semantic feature extractors to skin lesions  │
└──────────────────────────────────┬─────────────────────────────────────┘
                                   ▼
┌────────────────────────────────────────────────────────────────────────┐
│  Phase 3: Deep End-to-End Fine-Tuning                                  │
│  - Backbone: Deep layers UNFROZEN (Batch Normalization kept frozen)    │
│  - Learning Rate: Ultra-fine (LR = 2e-5)                               │
│  - Goal: Harmonize global feature representations across the network   │
└────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
ALL_METRICS   = []
ALL_HISTORIES = {}


def train_model_phased(
    model_name: str,
    base_model,
    unfreeze_phase2: int,
    unfreeze_phase3: int,
    preprocess_layer=None,
):
    """
    Three-phase training:
      Phase 1 – Head only  (backbone frozen)
      Phase 2 – Partial unfreeze (last N layers)
      Phase 3 – Full backbone fine-tune
    """
    print(f'\n{"+"*70}')
    print(f'  🔵 Training: {model_name}')
    print(f'{"+"*70}')

    model     = build_classification_head(
        base_model, NUM_CLASSES, DROPOUT_RATE, L2_REG, name=model_name,
        preprocess_layer=preprocess_layer)
    histories = []

    # ── PHASE 1 — Frozen backbone ────────────────────────────────────────────
    print('\n📌 Phase 1 — Frozen backbone, warm up head')
    freeze_all(base_model)
    model.compile(
        optimizer=Adam(BASE_LR),
        loss=FocalLoss(gamma=2.0, alpha=0.25),
        metrics=[
            'accuracy',
            keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top2_accuracy')
        ]
    )
    cbs1, _ = make_callbacks(f'{model_name}_p1', patience_es=6)
    cbs1.append(cosine_lr_schedule(BASE_LR, EPOCHS_PHASE1))
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE1,
                   callbacks=cbs1, class_weight=CLASS_WEIGHT_DICT, verbose=1)
    histories.append(h1.history)

    # ── PHASE 2 — Partial unfreeze ───────────────────────────────────────────
    print(f'\n📌 Phase 2 — Partial unfreeze (last {unfreeze_phase2} layers)')
    unfreeze_layers(base_model, unfreeze_phase2)
    model.compile(
        optimizer=Adam(FINETUNE_LR),
        loss=FocalLoss(gamma=2.0, alpha=0.25),
        metrics=[
            'accuracy',
            keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top2_accuracy')
        ]
    )
    cbs2, _ = make_callbacks(f'{model_name}_p2', patience_es=7)
    cbs2.append(cosine_lr_schedule(FINETUNE_LR, EPOCHS_PHASE2))
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE2,
                   callbacks=cbs2, class_weight=CLASS_WEIGHT_DICT, verbose=1)
    histories.append(h2.history)

    # ── PHASE 3 — Full fine-tune ─────────────────────────────────────────────
    print(f'\n📌 Phase 3 — Full fine-tune (last {unfreeze_phase3} layers)')
    unfreeze_layers(base_model, unfreeze_phase3)
    model.compile(
        optimizer=AdamW(learning_rate=FULL_FT_LR, weight_decay=1e-5),
        loss=FocalLoss(gamma=2.0, alpha=0.25),
        metrics=[
            'accuracy',
            keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top2_accuracy')
        ]
    )
    cbs3, _ = make_callbacks(f'{model_name}_p3', patience_es=8)
    cbs3.append(cosine_lr_schedule(FULL_FT_LR, EPOCHS_PHASE3))
    h3 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE3,
                   callbacks=cbs3, class_weight=CLASS_WEIGHT_DICT, verbose=1)
    histories.append(h3.history)

    # ── Save ─────────────────────────────────────────────────────────────────
    final_path = str(SAVE_DIR / f'{model_name}_final.keras')
    model.save(final_path)
    print(f'\n💾 Model saved → {final_path}')

    # ── Evaluate ─────────────────────────────────────────────────────────────
    metrics = evaluate_model(model, test_ds, CLASS_NAMES, model_name)
    print(f'\n📊 {model_name} Test Results:')
    print(f'  Accuracy   : {metrics["accuracy"]:.4f}')
    print(f'  F1 (macro) : {metrics["f1_macro"]:.4f}')
    print(f'  AUC-ROC    : {metrics["auc_roc"]:.4f}')
    print(f'  Cohen κ    : {metrics["kappa"]:.4f}')
    print(f'  MCC        : {metrics["mcc"]:.4f}')
    print('\n' + classification_report(
        metrics['y_true'], metrics['y_pred'],
        target_names=CLASS_NAMES, zero_division=0))

    # ── Plots ────────────────────────────────────────────────────────────────
    plot_training_history(histories, model_name, PLOT_DIR)
    plot_confusion_matrix(metrics['y_true'], metrics['y_pred'],
                          CLASS_NAMES, model_name, PLOT_DIR)
    plot_roc_curves(metrics['y_true'], metrics['y_pred_prob'],
                    CLASS_NAMES, model_name, PLOT_DIR)
    plot_per_class_metrics(metrics['report'], CLASS_NAMES, model_name, PLOT_DIR)

    ALL_METRICS.append(metrics)
    ALL_HISTORIES[model_name] = histories

    # ── Cleanup ──────────────────────────────────────────────────────────────
    del model
    gc.collect()
    K.clear_session()

    return metrics


print('✅ Training function ready.')


## 14. Model Benchmark 1: EfficientNetV2-B0
* **Architecture:** EfficientNetV2-B0 (Tan & Le, 2021)
* **Design Philosophy:** Optimized for faster training speed and parameter efficiency using Fused-MBConv blocks in early stages and neural architecture search (NAS).
* **Pretraining:** ImageNet-1k (Progressive learning).

In [ ]:
efficientnet_base = EfficientNetV2B0(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_preprocessing=True
)
metrics_eff = train_model_phased(
    model_name='EfficientNetV2B0',
    base_model=efficientnet_base,
    unfreeze_phase2=100,
    unfreeze_phase3=200,
    preprocess_layer=get_preprocess_layer('EfficientNetV2B0'),  # None — include_preprocessing=True handles it
)
del efficientnet_base; gc.collect()


## 15. Model Benchmark 2: ResNet-50
* **Architecture:** ResNet-50 (He et al., 2016)
* **Design Philosophy:** Introduces identity shortcut connections (*residual learning*) to resolve the vanishing gradient problem, allowing robust optimization across 50 deep convolutional layers.
* **Pretraining:** ImageNet-1k.

In [ ]:
resnet_base = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
metrics_res = train_model_phased(
    model_name='ResNet50',
    base_model=resnet_base,
    unfreeze_phase2=50,
    unfreeze_phase3=100,
    preprocess_layer=get_preprocess_layer('ResNet50'),  # caffe-style BGR + mean-subtract
)
del resnet_base; gc.collect()


## 16. Model Benchmark 3: DenseNet-121
* **Architecture:** DenseNet-121 (Huang et al., 2017)
* **Design Philosophy:** Connects each layer to every subsequent layer in a feed-forward fashion (*dense connectivity*). Promotes maximum feature reuse, substantial parameter reduction, and smooth gradient flow—widely regarded as a premier baseline for medical imaging.

In [ ]:
densenet_base = DenseNet121(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
metrics_den = train_model_phased(
    model_name='DenseNet121',
    base_model=densenet_base,
    unfreeze_phase2=60,
    unfreeze_phase3=120,
    preprocess_layer=get_preprocess_layer('DenseNet121'),  # torch-style mean/std normalise
)
del densenet_base; gc.collect()


## 17. Model Benchmark 4: MobileNetV3-Large
* **Architecture:** MobileNetV3-Large (Howard et al., 2019)
* **Design Philosophy:** Combines hardware-aware NAS, lightweight depthwise separable convolutions, hard-swish non-linearities, and Squeeze-and-Excitation (SE) attention blocks.
* **Deployment Profile:** Ideal for low-latency clinical edge deployment and mobile point-of-care diagnostics.

In [ ]:
mobilenet_base = MobileNetV3Large(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_preprocessing=True
)
metrics_mob = train_model_phased(
    model_name='MobileNetV3Large',
    base_model=mobilenet_base,
    unfreeze_phase2=25,
    unfreeze_phase3=60,
    preprocess_layer=get_preprocess_layer('MobileNetV3Large'),  # None — include_preprocessing=True handles it
)
del mobilenet_base; gc.collect()


## 18. Quantitative Benchmark Leaderboard & Comparative Analysis
We consolidate the evaluation metrics across all trained architectures into a unified comparison table to identify strengths, weaknesses, and clinical viability.

In [ ]:
comparison_rows = []
for m in ALL_METRICS:
    comparison_rows.append({
        'Model'        : m['model'],
        'Accuracy'     : m['accuracy'],
        'F1 (Macro)'   : m['f1_macro'],
        'F1 (Weighted)': m['f1_weighted'],
        'AUC-ROC'      : m['auc_roc'],
        'Cohen κ'      : m['kappa'],
        'MCC'          : m['mcc'],
    })

df_compare = pd.DataFrame(comparison_rows).sort_values(
    'Accuracy', ascending=False).reset_index(drop=True)

print('\n' + '='*75)
print('  📊  FULL MODEL COMPARISON TABLE')
print('='*75)
print(df_compare.to_string(index=False))
print('='*75)

best_model_name = df_compare.iloc[0]['Model']
best_accuracy   = df_compare.iloc[0]['Accuracy']
print(f'\n🏆 Best Model: {best_model_name}  |  Accuracy: {best_accuracy:.4f}')

df_compare.to_csv(SAVE_DIR / 'model_comparison.csv', index=False)
print('💾 Comparison table saved.')

## 19. Multi-Metric Comparative Visual Analytics
We generate comparative bar charts across primary clinical diagnostic indicators:
* **Accuracy & Balanced Accuracy**
* **Macro F1-Score**
* **One-vs-Rest AUC-ROC**
* **Cohen's Kappa ($\kappa$) & Matthews Correlation Coefficient (MCC)**

In [ ]:
metrics_to_plot = ['Accuracy', 'F1 (Macro)', 'AUC-ROC', 'Cohen κ', 'MCC']
palette         = sns.color_palette('tab10', n_colors=len(df_compare))

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(5 * len(metrics_to_plot), 6))
for ax, metric in zip(axes, metrics_to_plot):
    bars = ax.barh(df_compare['Model'], df_compare[metric],
                   color=palette, edgecolor='k', linewidth=0.6)
    ax.set_title(metric, fontweight='bold', fontsize=12)
    ax.set_xlim(max(0, df_compare[metric].min() - 0.05), 1.0)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    for bar, val in zip(bars, df_compare[metric]):
        ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

plt.suptitle('Model Performance Comparison — Face Skin Disease Classification',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'model_comparison_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── Radar Chart ─────────────────────────────────────────────────────────────
radar_metrics = ['Accuracy', 'F1 (Macro)', 'F1 (Weighted)', 'AUC-ROC', 'Cohen κ']
N      = len(radar_metrics)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for i, row in df_compare.iterrows():
    values = [row[m] for m in radar_metrics] + [row[radar_metrics[0]]]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'])
    ax.fill(angles, values, alpha=0.05)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Radar Chart — Model Comparison', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'model_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparison plots saved.')

## . Multi-Model Performance-Weighted Soft Voting Ensemble
To achieve maximum diagnostic accuracy and reduce individual architecture variance, we combine predictions from all successfully trained models into a **Weighted Soft Voting Ensemble**.

### Mathematical Formulation
Given $M$ individual models where model $m$ achieves Macro F1-score $w_m$ on the validation set, the ensemble predicted probability vector $\hat{\mathbf{P}}(\mathbf{x})$ for an input image $\mathbf{x}$ is:

$$\hat{\mathbf{P}}(\mathbf{x}) = \frac{\sum_{m=1}^{M} w_m \cdot \mathbf{P}_m(\mathbf{x})}{\sum_{m=1}^{M} w_m}, \quad \hat{y} = \arg\max_k \hat{\mathbf{P}}(\mathbf{x})_k$$

This weights stronger, more reliable classifiers higher while allowing complementary feature representations (e.g. DenseNet's dense local connectivity vs EfficientNetV2/MobileNetV3's efficiency-oriented features) to refine edge decisions.


In [ ]:
model_files = list(SAVE_DIR.glob('*_final.keras'))
print(f'📦 Found {len(model_files)} saved models for ensemble.')

if len(model_files) >= 2:
    ensemble_models, model_names = [], []
    for mf in model_files:
        try:
            # CUSTOM_OBJECTS (ResNet50Preprocess, DenseNet121Preprocess) must be
            # passed alongside FocalLoss — otherwise checkpoints that use a
            # custom preprocessing layer fail to deserialize ("Unknown layer")
            # and get silently dropped from the ensemble.
            m = keras.models.load_model(
                str(mf), custom_objects={**CUSTOM_OBJECTS, 'FocalLoss': FocalLoss})
            ensemble_models.append(m)
            model_names.append(mf.stem)
            print(f'  ✅ Loaded: {mf.name}')
        except Exception as e:
            print(f'  ⚠️  Could not load {mf.name}: {e}')

    # ══════════════════════════════════════════════════════════════════════
    # STAGE 1 — VALIDATION WEIGHT ESTIMATION
    # Ensemble weights are derived exclusively from validation Macro F1
    # scores. Test labels are never seen until Stage 2, so they cannot
    # influence the weights in any way.
    # ══════════════════════════════════════════════════════════════════════
    y_true_val = []
    val_probs_list = [[] for _ in ensemble_models]
    for images, labels in val_ds:
        y_true_val.extend(labels.numpy())
        for idx, em in enumerate(ensemble_models):
            val_probs_list[idx].extend(em.predict(images, verbose=0))

    y_true_val        = np.array(y_true_val)
    val_probs_arrays  = [np.array(p) for p in val_probs_list]

    validation_macro_f1 = np.array([
        f1_score(y_true_val, np.argmax(p, axis=1), average='macro', zero_division=0)
        for p in val_probs_arrays
    ])

    weight_sum = validation_macro_f1.sum()
    if weight_sum <= 0:
        # Edge case: every model scored 0 macro F1 on validation — fall back
        # to equal weighting instead of producing NaNs.
        weights = np.full(len(ensemble_models), 1.0 / len(ensemble_models))
    else:
        weights = validation_macro_f1 / weight_sum

    print(f'\nValidation Macro F1:')
    for name, f1v in zip(model_names, validation_macro_f1):
        print(f'  {name:<20}: {f1v:.4f}')

    print(f'\nEnsemble weights:')
    for name, w in zip(model_names, weights):
        print(f'  {name:<20}: {w:.4f}')

    print(f'\nWeight source: VALIDATION SET')
    print(f'Final evaluation source: TEST SET')

    # ══════════════════════════════════════════════════════════════════════
    # STAGE 2 — FINAL TEST EVALUATION
    # Using the already-fixed validation-derived weights. Test predictions
    # are matched to the SAME model ordering (model_names / ensemble_models)
    # used to compute the weights above.
    # ══════════════════════════════════════════════════════════════════════
    y_true_ens = []
    probs_list = [[] for _ in ensemble_models]
    for images, labels in test_ds:
        y_true_ens.extend(labels.numpy())
        for idx, em in enumerate(ensemble_models):
            probs_list[idx].extend(em.predict(images, verbose=0))

    y_true_ens   = np.array(y_true_ens)
    probs_arrays = [np.array(p) for p in probs_list]

    ensemble_probs = sum(w * p for w, p in zip(weights, probs_arrays))
    y_pred_ens     = np.argmax(ensemble_probs, axis=1)

    ens_acc   = (y_pred_ens == y_true_ens).mean()
    ens_f1    = f1_score(y_true_ens, y_pred_ens, average='macro', zero_division=0)
    ens_kappa = cohen_kappa_score(y_true_ens, y_pred_ens)
    y_bin_ens = label_binarize(y_true_ens, classes=np.arange(NUM_CLASSES))
    try:
        ens_auc = roc_auc_score(y_bin_ens, ensemble_probs,
                                 average='macro', multi_class='ovr')
    except Exception:
        ens_auc = float('nan')

    print(f'\n🎯 ENSEMBLE Results:')
    print(f'  Accuracy   : {ens_acc:.4f}')
    print(f'  F1 (Macro) : {ens_f1:.4f}')
    print(f'  AUC-ROC    : {ens_auc:.4f}')
    print(f'  Cohen κ    : {ens_kappa:.4f}')
    print('\n' + classification_report(
        y_true_ens, y_pred_ens, target_names=CLASS_NAMES, zero_division=0))

    plot_confusion_matrix(y_true_ens, y_pred_ens, CLASS_NAMES, 'Ensemble', PLOT_DIR)
    plot_roc_curves(y_true_ens, ensemble_probs, CLASS_NAMES, 'Ensemble', PLOT_DIR)

    ens_row = pd.DataFrame([{
        'Model'        : 'Ensemble (Weighted)',
        'Accuracy'     : round(float(ens_acc), 4),
        'F1 (Macro)'   : round(float(ens_f1), 4),
        'F1 (Weighted)': float('nan'),
        'AUC-ROC'      : round(float(ens_auc), 4),
        'Cohen κ'      : round(float(ens_kappa), 4),
        'MCC'          : float('nan')
    }])
    df_compare_final = pd.concat([df_compare, ens_row], ignore_index=True).sort_values(
        'Accuracy', ascending=False)
    df_compare_final.to_csv(SAVE_DIR / 'model_comparison_with_ensemble.csv', index=False)
    print('\n📊 Final Comparison (including Ensemble):')
    print(df_compare_final[['Model', 'Accuracy', 'F1 (Macro)', 'AUC-ROC', 'Cohen κ']].to_string(index=False))

    # ══════════════════════════════════════════════════════════════════════
    # FINAL LEAKAGE AUDIT
    # ══════════════════════════════════════════════════════════════════════
    print('\n' + '=' * 60)
    print('ENSEMBLE LEAKAGE AUDIT')
    print('=' * 60)
    print(f'Weight source       : Validation set')
    print(f'Weight metric       : Macro F1')
    print(f'Final score source  : Test set')
    print(f'Test leakage        : NONE')
    print(f'Status               : PASS')
    print('=' * 60)

else:
    print('⚠️  Need at least 2 trained models for ensemble. Skipping.')

## 21. Clinical Decision Support & Model Selection Recommendation
Here, we synthesize the empirical findings:
1. **Top Standalone Model:** Identifies the single highest-performing backbone on macro metrics.
2. **Ensemble Gain:** Measures the delta improvements yielded by multi-architecture soft voting.
3. **Deployment Guidance:** Highlights lightweight candidates (MobileNetV3 / EfficientNetV2) for resource-constrained edge devices vs heavy ensembles for high-certainty clinical decision support.

In [ ]:
print('='*75)
print('  🏆  FINAL RECOMMENDATION')
print('='*75)

if len(ALL_METRICS) > 0:
    best = max(ALL_METRICS, key=lambda x: x['accuracy'])
    print(f"\n🥇 Best Single Model : {best['model']}")
    print(f"   Accuracy          : {best['accuracy']:.4f}")
    print(f"   F1 (Macro)        : {best['f1_macro']:.4f}")
    print(f"   AUC-ROC           : {best['auc_roc']:.4f}")
    print(f"   Cohen κ           : {best['kappa']:.4f}")
    print(f"   MCC               : {best['mcc']:.4f}")

    print(f"""
📌 Deployment Guidance:
  • Best Accuracy  → Ensemble (Weighted Soft Voting)
  • Best Speed     → MobileNetV3Large
  • Best Balance   → {best['model']}

📂 Saved in /kaggle/working/saved_models/:
  • <Model>_final.keras        — full trained model
  • <Model>_best.keras         — best checkpoint per phase
  • <Model>_training_log.csv   — epoch logs
  • model_comparison.csv
  • model_comparison_with_ensemble.csv
  • label_map.json
""")

# Save label map
label_map = {i: cls for i, cls in enumerate(CLASS_NAMES)}
with open(SAVE_DIR / 'label_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)
print(f'✅ Label map saved: {label_map}')
print('\n✅ Pipeline complete. Ready for deployment!')

## 22. Standalone Diagnostic Inference & Multi-Model Demonstration
This demonstration verifies end-to-end inference on real held-out test images:
* Loads unseen test samples as raw RGB images.
* Runs inference across every saved checkpoint (`*_final.keras`).
* Displays predicted condition, individual model confidence scores, and differential diagnostic ranking.

In [ ]:
from PIL import Image as PILImage

def predict_single_image(image_path: str, model_path: str, label_map: dict, img_size: int = 224):
    # CUSTOM_OBJECTS (ResNet50Preprocess, DenseNet121Preprocess) must be passed
    # alongside FocalLoss, as in the ensemble step above — otherwise
    # ResNet50/DenseNet121 checkpoints fail to load.
    model = keras.models.load_model(
        model_path, custom_objects={**CUSTOM_OBJECTS, 'FocalLoss': FocalLoss})

    # ─── PIL-based loading: handles TIFF/WebP/any misnamed format ────────────
    with PILImage.open(image_path) as pil_img:
        pil_img = pil_img.convert('RGB').resize((img_size, img_size), PILImage.BILINEAR)
        img_np  = np.array(pil_img, dtype=np.float32)   # no /255.0 here — the model applies its own normalisation

    img = tf.constant(img_np)
    img = tf.clip_by_value(img, 0.0, 255.0)
    img = tf.expand_dims(img, 0)   # add batch dimension → (1, H, W, 3)

    probs      = model.predict(img, verbose=0)[0]
    pred_idx   = int(np.argmax(probs))
    pred_class = label_map[pred_idx]
    confidence = float(probs[pred_idx])

    del model
    K.clear_session()

    return pred_class, confidence, probs


def plot_prediction(image_path, pred_class, confidence, probs, label_map, model_name, img_size=224):
    with PILImage.open(image_path) as pil_img:
        img_disp = np.array(pil_img.convert('RGB').resize((img_size, img_size), PILImage.BILINEAR))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img_disp)
    axes[0].set_title(f'{model_name}\nPredicted: {pred_class}  ({confidence:.2%})',
                       fontsize=11, fontweight='bold',
                       color='green' if confidence > 0.7 else 'orange')
    axes[0].axis('off')

    colors = ['green' if i == int(np.argmax(probs)) else 'steelblue' for i in range(len(label_map))]
    axes[1].bar(list(label_map.values()), probs, color=colors, edgecolor='k')
    axes[1].set_ylabel('Probability'); axes[1].set_title('Class Probabilities')
    axes[1].tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()


# ─── Demo inference ───────────────────────────────────────────────────────────
# Runs automatically on every trained model, using a real held-out test image
# and looping over every `*_final.keras` checkpoint in SAVE_DIR.
demo_image_path = X_test[0]
demo_true_label = CLASS_NAMES[y_test[0]]
print(f'🖼️  Demo image: {demo_image_path}')
print(f'    True label : {demo_true_label}\n')

model_files_demo = sorted(SAVE_DIR.glob('*_final.keras'))
if len(model_files_demo) == 0:
    print('⚠️  No trained models found yet in', SAVE_DIR, '— run the training cells first.')
else:
    for mf in model_files_demo:
        model_name = mf.stem.replace('_final', '')
        try:
            pred_class, confidence, probs = predict_single_image(
                image_path=demo_image_path,
                model_path=str(mf),
                label_map=label_map,
            )
            correct = '✅' if pred_class == demo_true_label else '❌'
            print(f'{correct} {model_name:<18} → {pred_class:<20} ({confidence:.2%})')
            plot_prediction(demo_image_path, pred_class, confidence, probs, label_map, model_name)
        except Exception as e:
            print(f'⚠️  {model_name}: could not run inference — {e}')

print('\n✅ Inference demo complete for every trained model.')
